# Week 5: Advanced Ensemble Algorithms
## Notebook 10: Quantile Regression Forests & Uncertainty Intervals

### 1. Conceptual Intuition & Theoretical Background
Standard Random Forests approximate the **conditional mean**:
$$\mathbb{E}[Y \mid X = x] = \frac{1}{T} \sum_{t=1}^{T} \hat{y}_t(x)$$
However, conditional mean predictions provide zero insight into prediction variance, skewness, or tail risk.

**Quantile Regression Forests (Meinshausen, 2006):**
Instead of taking the mean of leaf node predictions, Quantile Forests retain **all observed training samples** that land in each leaf.
The full conditional distribution function is:
$$F(y \mid X = x) = \sum_{i=1}^{n} w_i(x) \cdot \mathbb{I}(Y_i \le y)$$
Where weights $w_i(x)$ measure how frequently training point $i$ shares leaf nodes with query $x$ across all $T$ trees:
$$w_i(x) = \frac{1}{T} \sum_{t=1}^{T} \frac{\mathbb{I}(X_i \in R_{\ell(x, t)})}{|R_{\ell(x, t)}|}$$

The $\tau$-th quantile $Q_\tau(x) = \inf \{ y : F(y \mid X = x) \ge \tau \}$.

---

### 2. Scratch Python Implementation: Quantile Forest Engine

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

class QuantileRegressionForestScratch:
    def __init__(self, n_estimators=20, max_depth=5):
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.trees = []
        self.X_train = None
        self.y_train = None
        
    def fit(self, X, y):
        self.X_train = np.asarray(X)
        self.y_train = np.asarray(y)
        n_samples = len(X)
        self.trees = []
        
        np.random.seed(42)
        for _ in range(self.n_estimators):
            # Bootstrap sample
            boot_idx = np.random.choice(n_samples, size=n_samples, replace=True)
            tree = DecisionTreeRegressor(max_depth=self.max_depth, random_state=42)
            tree.fit(self.X_train[boot_idx], self.y_train[boot_idx])
            self.trees.append(tree)
            
        return self
        
    def predict_quantile(self, X_query, quantile=0.5):
        X_query = np.asarray(X_query)
        n_query = len(X_query)
        n_train = len(self.X_train)
        quantiles = np.zeros(n_query)
        
        for q_idx in range(n_query):
            # Accumulate sample weights for this query point
            weights = np.zeros(n_train)
            x_q = X_query[q_idx:q_idx+1]
            
            for tree in self.trees:
                # Leaf index for query point
                query_leaf = tree.apply(x_q)[0]
                # Leaf indices for all training points
                train_leaves = tree.apply(self.X_train)
                
                in_same_leaf = (train_leaves == query_leaf)
                leaf_size = np.sum(in_same_leaf)
                if leaf_size > 0:
                    weights[in_same_leaf] += 1.0 / leaf_size
                    
            weights /= np.sum(weights) # Normalize to probability distribution
            
            # Compute empirical weighted quantile
            sort_idx = np.argsort(self.y_train)
            sorted_y = self.y_train[sort_idx]
            cum_weights = np.cumsum(weights[sort_idx])
            
            target_idx = np.searchsorted(cum_weights, quantile)
            quantiles[q_idx] = sorted_y[min(target_idx, len(sorted_y) - 1)]
            
        return quantiles

# Generate Heteroscedastic Synthetic Data
np.random.seed(42)
X_data = np.linspace(1, 10, 150).reshape(-1, 1)
noise = np.random.normal(0, 0.5 * X_data.squeeze(), size=150)
y_data = 2.0 * X_data.squeeze() + noise

qrf = QuantileRegressionForestScratch(n_estimators=15, max_depth=4).fit(X_data, y_data)

q10 = qrf.predict_quantile(X_data, quantile=0.10)
q50 = qrf.predict_quantile(X_data, quantile=0.50)
q90 = qrf.predict_quantile(X_data, quantile=0.90)

empirical_coverage = np.mean((y_data >= q10) & (y_data <= q90)) * 100

print("--- Quantile Regression Forest Results ---")
print(f"Empirical Coverage of [P10, P90]: {empirical_coverage:.1f}% (Expected ~80%)")
print(f"Prediction Interval Width at X=2:  {q90[15] - q10[15]:.2f}")
print(f"Prediction Interval Width at X=9:  {q90[-15] - q10[-15]:.2f}")
print("Intervals widen dynamically as heteroscedastic variance expands!")

### 3. Step-by-Step Code Walkthrough
1. **Leaf Membership Tracking:** Computes leaf assignments for both query and training instances via `tree.apply()`.
2. **Normalized Weight Accumulation:** Weights training samples inversely proportional to the leaf size they share with the query.
3. **Cumulative CDF Inversion:** Inverts cumulative sample weights to extract arbitrary quantiles.

---

### 4. Industry Applications & Trade-offs
- **Supply Chain Safety Stock:** Predicting the 95th percentile peak demand rather than mean expected demand.
- **Trade-off:** High inference memory overhead because the model must retain all training target values in memory.

---

### 5. Top Interview Questions & Answers
**Q1: How does Quantile Regression Forests differ from building separate Quantile Regressors for each quantile?**
> **Answer:** Standard quantile regression requires retraining a separate model for each quantile $\tau$. Quantile Regression Forests train a single model that reconstructs the full conditional distribution $F(y \mid X)$, enabling extraction of any quantile at query time.

**Q2: What guarantees that Quantile Regression Forest intervals are non-crossing?**
> **Answer:** Because predictions are computed by inverting a monotonically non-decreasing empirical cumulative distribution function ($F(y \mid X)$), predicted quantiles are guaranteed to satisfy $Q_{\tau_1} \le Q_{\tau_2}$ whenever $\tau_1 \le \tau_2$.